## Reading files from ADLSgen2

In [0]:
# dbutils.widgets.text("Table_Schema", "")
# dbutils.widgets.text("Table_Name", "")

# tableName = dbutils.widgets.get("Table_Name").lower()
# tableSchema = dbutils.widgets.get("Table_Schema").lower()

# def read_all_tables():
#     path = f"abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/OnpermSql_incremental_Loads/{tableSchema}/{tableSchema}_{tableName}/{tableName}.csv"
#     files = dbutils.fs.ls(path)
    
#     table_dfs = {}
#     for file in files:
#         if file.name.endswith('.csv'):
#             df = (spark.read.format("csv")
#                   .option("header", True)
#                   .load(file.path))
#             table_dfs[file.name] = df
#             df.show()   
            
#     return table_dfs
# read_all_tables()


## AutoLoader 

In [0]:

# dbutils.widgets.text("Table_Schema", "")
# dbutils.widgets.text("Table_Name", "")

# tableName = dbutils.widgets.get("Table_Name").lower()
# tableSchema = dbutils.widgets.get("Table_Schema").lower()

# print(f"Processing: {tableSchema}.{tableName}")


# spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")


# catalog = "eus-sales-catalog"

# schema = "bronze-layer"


# target_table = f"`{catalog}`.`{schema}`.{tableName}"

# print(f"Target table: {target_table}")

# target_path = f"abfss://bronze-layer@steusadlesgen0526.dfs.core.windows.net/"

# base_path = "abfss://landing-layer@steusadlesgen0526.dfs.core.windows.net/"

# source_path = f"{base_path}/{tableSchema}/{tableSchema}_{tableName}/"
# checkpoint_path = f"{target_path}/{catalog}/_checkpoints/{tableSchema}/{tableName}/"
# schema_path = f"{target_path}/{catalog}/_schemas/{tableSchema}/{tableName}/"

# print(f"Source path: {source_path}")


# df = (
#     spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "csv")
#     .option("cloudFiles.schemaLocation", schema_path)
#     .option("cloudFiles.inferColumnTypes", "true")
#     .option("cloudFiles.schemaEvolutionMode", "rescue")  
#     .option("header", "true")
#     .load(source_path)
# )

# from pyspark.sql.functions import current_timestamp, input_file_name

# df = (
#     df.withColumn("ingestion_time", current_timestamp())
     
# )

# (
#     df.writeStream
#     .format("delta")
#     .option("checkpointLocation", checkpoint_path)
#     .option("mergeSchema", "true")   
#     .outputMode("append")
#     .trigger(once=True) 
#     .toTable(target_table)
# )

In [0]:
# # Databricks Bronze DIM notebook 

# # ==============================
# # Widgets (ADF Parameters)
# # ==============================
# dbutils.widgets.text("Table_Schema", "")
# dbutils.widgets.text("Table_Name", "")
# dbutils.widgets.text("Watermark", "")

# tableName = dbutils.widgets.get("Table_Name").lower()
# tableSchema = dbutils.widgets.get("Table_Schema").lower()
# watermark_value = dbutils.widgets.get("Watermark")

# print(f"Processing: {tableSchema}.{tableName}")

# # ==============================
# # Config
# # ==============================
# spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
# spark.conf.set("spark.sql.files.ignoreMissingFiles", "true")

# catalog = "eus-sales-catalog"
# schema = "bronze-layer"

# # EXTERNAL LOCATION SETUP
# external_root = "abfss://bronze-layer@steusadlesgen0526.dfs.core.windows.net"
# target_external_path = f"{external_root}/{tableSchema}/{tableName}"

# # SOURCE AND METADATA PATHS (keep metadata OUTSIDE the target table path)
# base_landing_path = "abfss://landing-layer@steusadlesgen0526.dfs.core.windows.net/onpermdata"
# source_path = f"{base_landing_path}/{tableSchema}/{tableName}"
# metadata_root = f"{external_root}/_streaming_metadata"
# checkpoint_path = f"{metadata_root}/{tableSchema}/{tableName}/_checkpoints/"
# schema_path = f"{metadata_root}/{tableSchema}/{tableName}/_schemas/"

# print(f"Source path: {source_path}")
# print(f"Target external path: {target_external_path}")

# # ==============================
# # Create Schema
# # ==============================
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

# target_table = f"`{catalog}`.`{schema}`.`{tableName}`"
# print(f"Target table: {target_table}")


# # ==============================
# # Read Data with Auto Loader (from CSV files)
# # ==============================
# df = (
#     spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "csv")
#     .option("cloudFiles.schemaLocation", schema_path)
#     .option("cloudFiles.inferColumnTypes", "true")
#     .option("cloudFiles.schemaEvolutionMode", "rescue")
#     .option("header", "true")
#     .load(source_path)
# )

# from pyspark.sql.functions import current_timestamp, col, max as spark_max, lit
# from pyspark.sql.types import TimestampType

# # Add ingestion timestamp
# df = df.withColumn("ingestion_time", current_timestamp())

# # ==============================
# # Watermark Container
# # ==============================
# watermark_container = []
# table_created = False


# def process_batch(batch_df, batch_id):
#     global table_created

#     if not batch_df.take(1):
#         print(f"Batch {batch_id}: No data")
#         return

#     filtered_df = batch_df  # default — no filtering

#     if watermark_value and "max_incremental_value" in batch_df.columns:
#         filtered_df = batch_df.filter(        # filter is INSIDE the if block
#             col("max_incremental_value").cast(TimestampType()) >
#             lit(watermark_value).cast(TimestampType())
#         )

#     row_count = filtered_df.count()
#     print(f"Batch {batch_id}: Writing {row_count} rows")

#     # Write first
#     filtered_df.write \
#         .format("delta") \
#         .mode("append") \
#         .option("mergeSchema", "true") \
#         .save(target_external_path)

#     # Capture watermark only AFTER successful write
#     if "max_incremental_value" in batch_df.columns:
#         max_wm = batch_df.agg({"max_incremental_value": "max"}).collect()[0][0]
#         if max_wm:
#             watermark_container.append(str(max_wm))

#     if not table_created:
#         spark.sql(f"""
#             CREATE TABLE IF NOT EXISTS {target_table}
#             USING DELTA
#             LOCATION '{target_external_path}'
#         """)
#         table_created = True
        

#     # Write to Delta Bronze (external location)
#     row_count = filtered_df.count()
#     print(f"Batch {batch_id}: Writing {row_count} rows to {target_external_path}")
#     (
#         filtered_df.write
#         .format("delta")
#         .mode("append")
#         .option("mergeSchema", "true")
#         .save(target_external_path)
#     )
    
#     # Register external table after first write (only once)
#     if not table_created:
#         spark.sql(f"""
#             CREATE TABLE IF NOT EXISTS {target_table}
#             USING DELTA
#             LOCATION '{target_external_path}'
#         """)
#         table_created = True
#         print(f"External table registered: {target_table}")

# # ==============================
# # Start Stream (Micro-batch)
# # ==============================
# query = (
#     df.writeStream
#     .foreachBatch(process_batch)
#     .option("checkpointLocation", checkpoint_path)
#     .trigger(availableNow=True)
#     .start()
# )

# query.awaitTermination()

# # ==============================
# # Return Watermark to ADF
# # ==============================
# default_safety_date = "1900-01-01 00:00:00"

# final_output = watermark_container[-1] if watermark_container else default_safety_date

# print(f"[INFO] Returning watermark: {final_output}")

# dbutils.notebook.exit(final_output)

# ==============================
# Widgets (ADF Parameters)
# ==============================
dbutils.widgets.text("Table_Schema", "")
dbutils.widgets.text("Table_Name", "")
dbutils.widgets.text("Watermark", "")

tableName = dbutils.widgets.get("Table_Name").lower()
tableSchema = dbutils.widgets.get("Table_Schema").lower()
watermark_value = dbutils.widgets.get("Watermark")

print(f"Processing: {tableSchema}.{tableName}")

# ==============================
# Imports
# ==============================
from pyspark.sql.functions import current_timestamp, col, lit
from pyspark.sql.types import TimestampType

# ==============================
# Config
# ==============================
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set("spark.sql.files.ignoreMissingFiles", "true")

catalog = "eus-sales-catalog"
schema = "bronze-layer"

external_root = "abfss://bronze-layer@steusadlesgen0526.dfs.core.windows.net"
target_external_path = f"{external_root}/{tableSchema}/{tableName}"

base_landing_path = "abfss://landing-layer@steusadlesgen0526.dfs.core.windows.net/onpermdata"
source_path = f"{base_landing_path}/{tableSchema}/{tableName}"

metadata_root = f"{external_root}/_streaming_metadata"
checkpoint_path = f"{metadata_root}/{tableSchema}/{tableName}/_checkpoints/"
schema_path = f"{metadata_root}/{tableSchema}/{tableName}/_schemas/"

target_table = f"`{catalog}`.`{schema}`.`{tableName}`"

print(f"Source path:         {source_path}")
print(f"Target Delta path:   {target_external_path}")
print(f"Target table:        {target_table}")

# ==============================
# Create Schema if not exists
# ==============================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

# ==============================
# Read with Auto Loader
# ==============================
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .load(source_path)
)

df = df.withColumn("ingestion_time", current_timestamp())

# ==============================
# State containers
# ==============================
watermark_container = []
table_created = False

# ==============================
# foreachBatch function
# ==============================
def process_batch(batch_df, batch_id):
    global table_created

    # Skip empty batches
    if not batch_df.take(1):
        print(f"Batch {batch_id}: No data, skipping")
        return

    # ── Filter to only new rows based on watermark ──
    filtered_df = batch_df

    if watermark_value and "max_incremental_value" in batch_df.columns:
        filtered_df = batch_df.filter(
            col("max_incremental_value").cast(TimestampType()) >
            lit(watermark_value).cast(TimestampType())
        )

    row_count = filtered_df.count()
    print(f"Batch {batch_id}: {row_count} rows to write")

    if row_count == 0:
        print(f"Batch {batch_id}: Nothing new after watermark filter, skipping write")
        return

    # ── Write to Delta Bronze ──
    filtered_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .save(target_external_path)

    print(f"Batch {batch_id}: Write complete → {target_external_path}")

    # ── Capture watermark AFTER successful write ──
    if "max_incremental_value" in batch_df.columns:
        max_wm = batch_df.agg({"max_incremental_value": "max"}).collect()[0][0]
        if max_wm:
            watermark_container.append(str(max_wm))
            print(f"Batch {batch_id}: Watermark captured → {max_wm}")

    # ── Register external table once ──
    if not table_created:
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {target_table}
            USING DELTA
            LOCATION '{target_external_path}'
        """)
        table_created = True
        print(f"Batch {batch_id}: External table registered → {target_table}")

# ==============================
# Start Stream
# ==============================
query = (
    df.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

# ==============================
# Return max watermark to ADF
# ==============================
default_safety_date = "1900-01-01 00:00:00"

final_output = max(watermark_container) if watermark_container else default_safety_date

print(f"[INFO] Returning watermark to ADF: {final_output}")

dbutils.notebook.exit(final_output)

In [0]:
# files = dbutils.fs.ls('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/')
# display(files)

In [0]:
# df_titles = spark.read.format('csv').option('header','true').load('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/netflix_titles.csv')
# # display(df_titles)
# df_cast = spark.read.format('csv').option('header','true').load('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/netflix_cast.csv')
# # display(df_cast)
# df_country = spark.read.format('csv').option('header','true').load('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/netflix_countries.csv')
# # display(df_country)
# df_director = spark.read.format('csv').option('header','true').load('abfss://landing-layer@stadlsgen2eus0426.dfs.core.windows.net/Netflix_files/netflix_directors.csv')
# display(df_director)

In [0]:
# df_titles.write.saveAsTable("`adb-netflix`.bronze-layer.netflix_titles_managed")
# df_cast.write.saveAsTable("`adb-netflix`.bronze-layer.netflix_cast_managed")
# df_country.write.saveAsTable("`adb-netflix`.bronze-layer.netflix_country_managed")
# df_director.write.saveAsTable("`adb-netflix`.bronze-layer.netflix_director_managed")

In [0]:
# from pyspark.sql.functions import col, current_timestamp

# # Step 1: Replace date_added with current timestamp for all rows
# df_titles = df.withColumn("date_added", current_timestamp())

# # Step 2: Select and cast columns properly
# df_titles = df_titles.select(
#     col("show_id").cast("bigint"),
#     col("title").cast("string"),
#     col("type").cast("string"),
#     col("release_year").cast("int"),
#     col("rating").cast("string"),
#     col("date_added").cast("timestamp"),
#     col("description").cast("string"),
#     col("duration_minutes").cast("int"),
#     col("duration_seasons").cast("int")
# )

# # Step 3: Directors dataframe
# df_directors = df_director.select(
#     col("director").cast("string"),
#     col("show_id").cast("bigint")
# )

# Step 4: Display results
# display(df_titles)
# display(df_directors)

In [0]:
# df_titles.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`adb-netflix`.`bronze-layer`.`netflix_titles`")

# df_cast.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     "`adb-netflix`.`bronze-layer`.`netflix_cast`"
# )
# df_country.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     "`adb-netflix`.`bronze-layer`.`netflix_country`"
# )
# df_director.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     "`adb-netflix`.`bronze-layer`.`netflix_director`")
# # )

## Writing into bronze-Schema

In [0]:
# table_dfs = read_all_tables()
# df = list(table_dfs.values())[0]

# if tableSchema == "netflix":
#     catalog = "adb-netflix"
# else:
#     catalog = "adv2019"

# schema = "bronze-layer"


# target_table = f"`{catalog}`.`{schema}`.{tableName}"

# df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(target_table)